In [1]:
import sys
import os
import csv
from gpt_suite import gpt_mp_handler
import pandas as pd
import random
import json
from tqdm.auto import tqdm
import re
from datetime import datetime

In [2]:
norms_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_data.csv', sep=',')
dialogues_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/dialogues_data.csv', sep=',')
themes_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/theme_definitions_v4.csv', sep=',')

In [3]:
norms_df[:3]

,id_,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,dialogue_id,theme_id
0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,0,16
1,1,2. Unity within the family: Maintaining harmon...,0.1131,False,False,[],[],24,24,0,14
2,2,3. Social relationships and obligations: Chine...,0.0935,False,False,[],[],24,24,0,14


In [4]:
dialogues_df[:3]

,id_,identifier,culture,dialogue_string,display_text,summary,other_features
0,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{}
1,1,mpdd-2,mandarin-chinese,"Cousin: Zheng Peng, I heard from the villagers...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zheng Peng's cousin info...",{}
2,2,mpdd-3,mandarin-chinese,"Zuo Zhengpeng: Mom, today is Sunday and I want...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zuo Zhengpeng informs hi...",{}


In [5]:
themes_df[-3:]

,id_,name,description,violation_characteristic,activation_settings,actors,recepients
42,44,RespectForTeachers,"Respecting their knowledge, guidance, and auth...","Disrespecting teachers' authority, disregardin...","school, educational settings, any public settings",any,"teacher, educator"
43,45,SchedulingMeetingTimesInAdvance,Scheduling meeting times in advance shows punc...,"Changing meeting times frequently, arriving la...",any settings,any,any
44,46,ConcernForOthers,Showing concern for others' discomfort or trou...,"Ignoring the troubles of others, not listening...",any settings,any,any


In [6]:
def extract_dialogue_text(display_text):
    display_text = display_text.replace('<br>', '\n').replace('<b>', '').replace('</b>', '')
    dialogue_text_zh = '\n'.join(display_text.split('Dialogue & Translation:')[1].strip().split('\n')[::3])
    return dialogue_text_zh

dialogues_df['dialogue_text_zh'] = dialogues_df['display_text'].apply(extract_dialogue_text)

dialogues_df[:3]

,id_,identifier,culture,dialogue_string,display_text,summary,other_features,dialogue_text_zh
0,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...
1,1,mpdd-2,mandarin-chinese,"Cousin: Zheng Peng, I heard from the villagers...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zheng Peng's cousin info...",{},堂弟(surprise): 正鵬哥，聽村上人講昨天你娶了個嫂子？\n左正鵬(disgust)...
2,2,mpdd-3,mandarin-chinese,"Zuo Zhengpeng: Mom, today is Sunday and I want...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zuo Zhengpeng informs hi...",{},左正鵬(neutral): 媽，今天是星期天我想去趕下集，可能下午才能回家，有什麼事就別等我...


In [7]:
norm_dialogues_df = pd.merge(norms_df, dialogues_df, left_on='dialogue_id', right_on='id_', how='inner')
print(len(norms_df))
print(len(norm_dialogues_df))
norm_dialogues_df[:3]

63779
63779


,id__x,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,dialogue_id,theme_id,id__y,identifier,culture,dialogue_string,display_text,summary,other_features,dialogue_text_zh
0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,0,16,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...
1,1,2. Unity within the family: Maintaining harmon...,0.1131,False,False,[],[],24,24,0,14,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...
2,2,3. Social relationships and obligations: Chine...,0.0935,False,False,[],[],24,24,0,14,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...


In [8]:
norm_dialogues_themes_df = pd.merge(norm_dialogues_df, themes_df, left_on='theme_id', right_on='id_', how='inner')
print(len(norm_dialogues_df))
print(len(norm_dialogues_themes_df))
norm_dialogues_themes_df[:3]

63779
63779


,id__x,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,dialogue_id,...,summary,other_features,dialogue_text_zh,id_,name,description,violation_characteristic,activation_settings,actors,recepients
0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,0,...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent
1,15,2. Filial piety,0.2567,False,False,[],[],24,24,5,...,"In this conversation, 徐父 (Mr. Xu) displays dis...",{},徐父(angry): 是嗎，舊的不爛，新的不會來嘛！我還要繼續砸呢！\n徐父(angry):...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent
2,18,2. Filial piety: Children are expected to prio...,0.0858,False,False,[],[],24,24,6,...,"In this conversation, Zheng Peng informs his m...",{},左母(surprise): 正鵬，今天你到那個憨同學家裏，她父母跟你說了些什麼？她自己呢？\...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent


In [9]:
def camel_to_phrase(camel_str):
    # Insert a space before each uppercase letter, except the first one
    phrase = re.sub(r'(?<!^)(?=[A-Z])', ' ', camel_str)
    # Capitalize the first letter and make the rest lowercase
    return phrase.capitalize()

print(camel_to_phrase('respectForAuthority'))

Respect for authority


In [10]:
def make_symbolic_prompts(row):
    prompt = f"""Norm ID: {row['id__x']}
Theme ID: {row['id_']}

Conversation from Chinese culture:
{row['dialogue_text_zh'].strip()}

Social Norm:\n{row['text'].strip()}

Norm Concept Name: {camel_to_phrase(row['name'].strip())}
Norm Concept Description: {row['description'].strip()}
Norm Concept Potential Violation Sketch: {row['violation_characteristic'].strip()}
Norm Concept Applicability Scenario: {row['activation_settings'].strip()}
Norm Concept Enactor Role: {row['actors'].strip()}
Norm Concept Acceptor Role: {row['recepients'].strip()}
""".strip()
    return prompt

norm_dialogues_themes_df['symbolic_prompt'] = norm_dialogues_themes_df.apply(make_symbolic_prompts, axis=1)

print(norm_dialogues_themes_df[:1].symbolic_prompt.iloc[0])

Norm ID: 0
Theme ID: 16

Conversation from Chinese culture:
左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！
左父(surprise): 哎喲，老婆子，你怎麼盡講那些不利於團結的話呢！他去送送他的同學也在情理之中嘛！
左正鵬(neutral): 爸、媽，我回來啦！
左母(disgust): 我怕你喝了迷魂湯，魂被你那個憨同學勾引去了！
左父(surprise): 老婆子，看你講話像個母親說的嗎！你怎麼老是反對他們倆的事呢？
左正鵬(neutral): 爸、媽，你們都不要講了。這件事我自己有主見的，我知道自己該怎麼去做，不應該怎麼不去做。決不會胡來的，請您們放心好了！

Social Norm:
1. Respect for parents: Filial piety and respect for parents are highly valued in Chinese culture. Children are expected to listen to and obey their parents' opinions and decisions.

Norm Concept Name: Care for parents
Norm Concept Description: Children are expected to care for parents in their old age
Norm Concept Potential Violation Sketch: Neglecting a parent in need or ignoring duty towards parent(s)
Norm Concept Applicability Scenario: family
Norm Concept Enactor Role: child
Norm Concept Acceptor Role: parent


In [11]:
openai_key = '<put-your-key-here>'
# openai_key = '<put-your-key-here>'
# openai_key = '<put-your-key-here>'

In [12]:
debug_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_verification_logs/'
os.makedirs(debug_dir, exist_ok=True)

In [13]:
# DEFAULT_SYSTEM_PROMPT = """You are a helpful assistant. Your task is to make judgments and provide annotations for a given conversation from Chinese culture, a social norm and a given norm concept.
# Consider factors such as age of the people involved, relationships between them, settings of the conversations such as work, family or friends, topic of conversation and so on.
# Your task is to first judge whether the given social norm matches the provided norm concept category. Provide a justification for this judgment.
# Then provide a judgement for whether the social norm and norm concept are relevant for the given conversation. Provide a justification for this judgment as well.
# Then you need to provide a judgment of whether this social norm was violated in the provided conversation. Provide a justification for this judgment.
# Then, you need to annotate the social norm with conversation specific details such as enactor and acceptor roles. Provide a single phrase for each detail. You may say unsure if it is impossible to observe from the conversation. These details are defined as:
# Enactor Role - Social position of the person whose actions are expected to adhere to the provided social norm.
# Acceptor Role - Social position of the person who is experiencing the consequences of the adherence or violation of the norm in the provided situation.
# If there was a violation of the social norm, provide a short one sentence description of how the social norm was violated in this situation.
# Then, if there is a violation provide the following extra details:
# Violating Action - A short phrase describing the action which caused the violation (e.g. badmouthing parents, use of abusive language, etc.)
# Violator Role - This should ideally match the provided enactor role of the social norm above
# Victim Role - This should ideally match the provided acceptor role of the social norm above
# Violator Emotion - Emotion of the violator when they caused the violation - respond with one of 9 basic plutchik categories for emotion
# Victim Emotion - Emotion of the victim right after they experienced the violation - respond with one of 9 basic plutchik categories for emotion
# Only provide violating action, violator role, victim role, violator emotion and victim emotion when a violation occurs.
# Your response should strictly follow the following format:
# Social Norm - Norm Concept Compatibility: <match/doesn't match>
# Compatibility Justiciation: <short justification>

# {Only If compatible}
# Relevance: <relevant/irrelevant>
# Relevance Justification: <short justification>

# Enactor Role: <short phrase. e.g: parent/child/neighbor/boss/employee and so on.>
# Acceptor Role: <short phrase. e.g: parent/child/neighbor/boss/employee and so on.>

# Violation Status: <adhere/violate>
# Violation Status Justification: <short justification>

# {Only if a violation occurs}
# Violating Action: <short phrase>
# Violator Role: <short phrase. e.g: parent/child/neighbor/boss/employee and so on.>
# Victim Role: <short phrase. e.g: parent/child/neighbor/boss/employee and so on.>
# Violator Emotion: <anger/fear/sadness/disgust/surprise/anticipation/trust/joy/neutral>
# Victim Emotion: <anger/fear/sadness/disgust/surprise/anticipation/trust/joy/neutral>
# """

DEFAULT_SYSTEM_PROMPT = """
### Task Overview:
As a cultural and social norms analysis assistant, your task is to evaluate a provided conversation from the context of Chinese culture based on a given social norm and a corresponding norm concept. Your analysis should be comprehensive, considering factors such as the age, relationships, settings (e.g., work, family, friends), and the topic of the conversation.

### Steps for Analysis:

1. **Evaluate Social Norm and Norm Concept Compatibility**  
   - **Task:** Judge whether the provided social norm aligns with the given norm concept.
   - **Action:** State if the social norm **matches** or **doesn't match** the norm concept.
   - **Justification:** Provide a concise reason for your judgment.

2. **Assess Relevance to the Conversation**  
   - **Task:** Determine if the provided social norm and norm concept are relevant to the context of the conversation.
   - **Action:** State if the norm is **relevant** or **irrelevant**.
   - **Justification:** Provide a concise reason for your judgment.
  
3. **Determine Social Norm Violation**  
   - **Task:** Judge whether the social norm was **adhered to** or **violated** in the conversation.
   - **Justification:** Provide a concise reason for your judgment.
  
4. **Annotate Conversation-Specific Details**  
   - **Enactor Role:** Identify the social role of the person expected to follow the norm (e.g., parent, child, boss). Strictly do not provide names of the speakers here. Provide the specific social role of the enactor in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.
   - **Acceptor Role:** Identify the social role of the person affected by the norm's adherence or violation (e.g., parent, child, employee). Strictly do not provide names of the speakers here. Provide the specific social role of the acceptor in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.

5. **Violation Analysis** *(Only if a violation occurred)*  
   If the norm was violated, provide the following additional details:
   - **Violating Action:** A brief description of the action that caused the violation (e.g., "badmouthing parents").
   - **Violator Role:** Social role of the person who violated the norm. Strictly do not provide names of the speakers here. Provide the specific social role of the violator in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.
   - **Victim Role:** Social role of the person affected by the violation. Strictly do not provide names of the speakers here. Provide the specific social role of the victim in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.
   - **Violator Emotion:** Identify the violator's emotion using one of the following: **anger, fear, sadness, disgust, surprise, anticipation, trust, joy, neutral**.
   - **Victim Emotion:** Identify the victim's emotion using one of the following: **anger, fear, sadness, disgust, surprise, anticipation, trust, joy, neutral**.

### Response Format:
Your response must adhere to the format below:

Social Norm - Norm Concept Compatibility: <match/doesn't match>
Compatibility Justification: <short justification>

{Only if compatible}
Relevance: <relevant/irrelevant>
Relevance Justification: <short justification>

Enactor Role: <sitatuion specific social role, strictly not the name, of the person>
Acceptor Role: <sitatuion specific social role, strictly not the name, of the person>

Violation Status: <adhere/violate>
Violation Status Justification: <short justification>

{Only if a violation occurs}
Violating Action: <short phrase>
Violator Role: <sitatuion specific social role, strictly not the name, of the person>
Victim Role: <sitatuion specific social role, strictly not the name, of the person>
Violator Emotion: <one of 9 basic emotions>
Victim Emotion: <one of 9 basic emotions>
""".strip()


follow_up_question = """
Reconsider the provided response and judge whether the provided annotations are accurate. Respond with a accurate/inaccurate label and a short justification for the decision.
### Response Format:
Your response must adhere to the format below:

Quality Judgment: <accurate/inaccurate>
Justification: <short justification>
""".strip()


def symbolic_annotator(symbolic_prompts) -> list:
    config = {"temperature": 0, "max_tokens": 500}
    model = 'gpt-4o-mini'
    handler = gpt_mp_handler.GPTMPHandler(api_key=openai_key, gen_conf=config, num_worker=10)
    results = list()
    batch = []
    for norm_id in symbolic_prompts:
        ins = {
            'init_context': '',
            'questions': [symbolic_prompts[norm_id], follow_up_question],
            'task_desc': DEFAULT_SYSTEM_PROMPT,
            'debug_log': debug_dir,
            'model_name': model
        }
        batch.append(ins)
        results.append([norm_id])
    handler.add_batch(batch)
    outs = handler.process()

    # print(outs)
    for idx, out in enumerate(outs):
        if len(out) == 0:
            print("ERROR:Missing...")
            results[idx].append(-1)
            continue
        for ques, resp in out.items():
            results[idx].append(resp)
    return results

In [14]:
norm_dialogues_themes_df_filtered = norm_dialogues_themes_df[~norm_dialogues_themes_df.name.str.contains('KMeans')]
print(len(norm_dialogues_themes_df_filtered))
norm_dialogues_themes_df_filtered[:3]

40816


,id__x,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,dialogue_id,...,other_features,dialogue_text_zh,id_,name,description,violation_characteristic,activation_settings,actors,recepients,symbolic_prompt
0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,0,...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent,Norm ID: 0\nTheme ID: 16\n\nConversation from ...
1,15,2. Filial piety,0.2567,False,False,[],[],24,24,5,...,{},徐父(angry): 是嗎，舊的不爛，新的不會來嘛！我還要繼續砸呢！\n徐父(angry):...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent,Norm ID: 15\nTheme ID: 16\n\nConversation from...
2,18,2. Filial piety: Children are expected to prio...,0.0858,False,False,[],[],24,24,6,...,{},左母(surprise): 正鵬，今天你到那個憨同學家裏，她父母跟你說了些什麼？她自己呢？\...,16,CareForParents,Children are expected to care for parents in t...,Neglecting a parent in need or ignoring duty t...,family,child,parent,Norm ID: 18\nTheme ID: 16\n\nConversation from...


In [15]:
symbolic_prompts = norm_dialogues_themes_df_filtered[['id__x', 'symbolic_prompt']].set_index('id__x')['symbolic_prompt'].to_dict()
print(len(symbolic_prompts))
for k, v in symbolic_prompts.items():
    toss = random.random()
    if toss >= 0.9:
        print(k)
        print(v)
        break

40816
263
Norm ID: 263
Theme ID: 16

Conversation from Chinese culture:
師母(neutral): 小左，你妻子在鄉下還好嗎？近段來你倆的關係怎樣？聽說你妻子在鄉下還要種田、帶小孩、孝敬公婆，想必很辛苦吧！
左正鵬(neutral): 師母，感謝你的關心，我妻子在家裏是很辛苦的。家裏的一切事務都落在她的肩上，她對我也很關心。去年她把家裏一頭大肥豬的錢寄來給我做生活費了。我倆的關係一直很好。前次她來過我們學校，她可能對我有點不放心，怕我在學校有外遇。所以我給她許下了心願：她的“滴水之恩”，我一定“涌泉相報”，我決不做千古罵名的“陳世美”。後來她也相信我在學校是最清白的，沒有外遇，正因爲這樣，我也沒有任何理由拋棄她，我也不願意接受其他任何女人的“愛”。
師母(surprise): 小左，今天我叫你來，也沒有別的事情可說的。主要是想瞭解下你家裏的情況，怕你妻子在家裏鬧情緒會影響你的學習。既然她對你很好我也放心了，師母也希望你倆同舟共濟，白頭偕老。你對妻子的一往情深，師母感到敬佩，敬佩呀！
左正鵬(neutral): 師母，做人就是要講良心、講道德，時刻要用倫理道德這把尺子衡量自己。我雖然不是什麼大人物，但是做人的基本原則我自己也懂得。您說是不是？
師母(neutral): 是啊！你說得在理，做人就應該這樣。
左正鵬(neutral): 師母，您對我這麼關心，我一定努力學習，以優異成績和對家庭負責任的態度來表達師母對我的關愛。
師母(neutral): 我剛纔說了這些，實際上也是我們長輩應該說的，你別在意啊！
左正鵬(neutral): 師母，您對我們每個同學應該都是父母之心吧，應該都是平等的。聽說我班楊偉同學正在追求劉豔，但是劉豔不接受他的愛，您能不能給劉豔做點思想工作，給他們倆搭撘橋，牽牽線呢？
師母(surprise): 啊!有這麼回事？有時間我去找找劉豔吧！
左正鵬(fear): 師母，您千萬別說是我講的啊！
師母(neutral): 就說是你講的也沒關係吧！反正“男大當婚，女大當嫁”嘛！一家養女，百家求吧！
左正鵬(neutral): 師母，您說得對！

Social Norm:
1. Filial piety: The expectation that individuals should r

In [24]:
json.dump([DEFAULT_SYSTEM_PROMPT, follow_up_question, symbolic_prompts], open('/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_prompts.json', 'w'))
norm_dialogues_themes_df.to_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_dialogues_themes_data.csv')

In [16]:
symbolic_prompt_keys = sorted(list(symbolic_prompts.keys()))

batch_id = 0
batches = []
b = 0
bsz = 1000
e = bsz
while b < len(symbolic_prompt_keys):
    batch_keys = symbolic_prompt_keys[b:e]
    batch = {}
    for bkey in batch_keys:
        batch[bkey] = symbolic_prompts[bkey]
    batches.append(batch)
    b += bsz
    e += bsz

print(len(batches))

41


In [17]:
output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_annotation_verification_outputs/'
os.makedirs(output_dir, exist_ok=True)

t1 = datetime.now()
for i, batch in tqdm(enumerate(batches)):
    batch_out_path = os.path.join(output_dir, f"batch_{i}.json")
    if not os.path.exists(batch_out_path):
        batch_res = symbolic_annotator(batch)
        json.dump(batch_res, open(batch_out_path, 'w'))
    t2 = datetime.now()
    print(f'batch {i} done.', t2-t1)

0it [00:00, ?it/s]

batch 0 done. 0:00:00.027413
batch 1 done. 0:00:00.027686
batch 2 done. 0:00:00.027881
batch 3 done. 0:00:00.028132
batch 4 done. 0:00:00.028383
batch 5 done. 0:00:00.028635
batch 6 done. 0:00:00.028889
batch 7 done. 0:00:00.029148
batch 8 done. 0:00:00.029394
batch 9 done. 0:00:00.029647
batch 10 done. 0:00:00.029897
batch 11 done. 0:00:00.030151
batch 12 done. 0:00:00.030407
batch 13 done. 0:00:00.030656
batch 14 done. 0:00:00.030923
batch 15 done. 0:00:00.031170
batch 16 done. 0:00:00.031419
batch 17 done. 0:00:00.031677
batch 18 done. 0:00:00.031934
batch 19 done. 0:00:00.032207
batch 20 done. 0:00:00.032469
batch 21 done. 0:00:00.032717
batch 22 done. 0:00:00.032971
batch 23 done. 0:00:00.033220
batch 24 done. 0:00:00.033479
batch 25 done. 0:00:00.033768
batch 26 done. 0:00:00.034055
batch 27 done. 0:00:00.034343
batch 28 done. 0:00:00.034616


Verifying Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 29 done. 0:09:04.611147


...

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 39 done. 1:48:13.314912


Verifying Batch:   0%|          | 0/816 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/816 [00:00<?, ?it/s]

batch 40 done. 1:56:14.044188


In [23]:
w = 0
t = 0
rem_batch = {}
ann_ids = set()
for i, batch in tqdm(enumerate(batches)):
    if os.path.exists(os.path.join(output_dir, f'batch_{i}.json')):
        batch_res = json.load(open(os.path.join(output_dir, f'batch_{i}.json')))
        for res in batch_res:
            if len(res) != 4 or res[1].strip() != '' or 'compatibility' not in res[2].lower() or 'quality' not in res[3].lower():
                w += 1
                rem_batch[res[0]] = symbolic_prompts[res[0]]
            else:
                ann_ids.add(res[0])
        t += len(batch_res)
print(w, t, len(rem_batch), len(ann_ids))

0it [00:00, ?it/s]

0 40816 0 40816
